# IBM Cloud

# Storage (Cos)

In [3]:
!pip install ibm-cos-sdk==2.14.0.

  Using cached ibm_cos_sdk-2.14.0-py3-none-any.whl
  Using cached ibm_cos_sdk_core-2.14.0-py3-none-any.whl
  Using cached ibm_cos_sdk_s3transfer-2.14.0-py3-none-any.whl
  Using cached jmespath-1.0.1-py3-none-any.whl.metadata (7.6 kB)
  Using cached requests-2.32.2-py3-none-any.whl.metadata (4.6 kB)
  Using cached urllib3-2.4.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached idna-3.10-py3-none-any.whl.metadata (10 kB)
  Using cached certifi-2025.4.26-py3-none-any.whl.metadata (2.5 kB)
Using cached jmespath-1.0.1-py3-none-any.whl (20 kB)
Using cached requests-2.32.2-py3-none-any.whl (63 kB)
Using cached urllib3-2.4.0-py3-none-any.whl (128 kB)
Using cached certifi-2025.4.26-py3-none-any.whl (159 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 198.8/198.8 kB 2.1 MB/s eta 0:00:00a 0:00:01
Using cached idna-3.10-py3-none-any.whl (70 kB)

[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import ibm_boto3
from ibm_botocore.client import Config, ClientError

# Constants for IBM COS values
COS_ENDPOINT = "https://s3.us-south.cloud-object-storage.appdomain.cloud"
COS_API_KEY_ID = "1IVfOmwvWWvLchoC2reuQNIoCC_s28ri3k3jQ30PI1XW"
COS_INSTANCE_CRN = "crn:v1:bluemix:public:cloud-object-storage:global:a/a3ddc697dfc74d418f1d2974f79fe20e:a106584d-8374-4867-ad78-7b73f6bf2966::"


In [ ]:

# Code Create by Eric
cos_client = ibm_boto3.client("s3",
    ibm_api_key_id=COS_API_KEY_ID,
    ibm_service_instance_id=COS_INSTANCE_CRN,
    verify=True,
    endpoint_url=COS_ENDPOINT,
    config=Config(signature_version="oauth"))

def get_buckets():
    print("Retrieving list of buckets")
    bucketList = []
    try:
        buckets = cos_client.list_buckets()
        for bucket in buckets["Buckets"]:
            bucketList.append(bucket["Name"])
        return bucketList
    except ClientError as be:
        print("Client error: {0}\n".format(be))
    except Exception as e:
        print("Unable to retrieve list buckets: {0}".format(e))

def get_object(bucket_name, item_name):
    print("Retrieving item from bucket: {0}, key: {1}".format(bucket_name, item_name))
    try:
        file = cos_client.get_object(Bucket=bucket_name, Key=item_name)
        return file["Body"].read()
    except ClientError as be:
        print("Client error: {0}\n".format(be))
    except Exception as e:
        print("Unable to retrieve file contents: {0}".format(e))

def get_bucket_contents():
    print("Retrieving bucket contents from: {0}".format('bucket-synapse-iq'))
    bucketContentList = []
    try:
        files = cos_client.list_objects(Bucket='bucket-synapse-iq')
        for file in files.get("Contents", []):
            bucketContentList.append((file["Key"]))
        return bucketContentList
    except ClientError as be:
        print("Client error: {0}\n".format(be))
    except Exception as e:
        print("Unable to retrieve bucket contents: {0}".format(e))

### Class: IBMCOSManager

In [40]:
from io import BytesIO
import ibm_boto3
from ibm_botocore.client import Config, ClientError

In [29]:

## Class by connecting to IBM Cloud Object Storage
## This class is a singleton that manages the connection to IBM Cloud Object Storage (COS) and provides methods for common operations.

import os
from io import BytesIO
import ibm_boto3
from ibm_botocore.client import Config, ClientError
from threading import Lock

class IBMCOSManager:
    _instance = None
    _lock = Lock()

    def __new__(cls, *args, **kwargs):
        # Singleton pattern: ensures only one instance is created
        if not cls._instance:
            with cls._lock:
                if not cls._instance:
                    cls._instance = super(IBMCOSManager, cls).__new__(cls)
        return cls._instance

    def __init__(self,
                 api_key=None,
                 service_crn=None,
                 endpoint=None,
                 verify_ssl=True):

        # Prevents re-initialization of the singleton instance
        if hasattr(self, "_initialized") and self._initialized:
            return

        self.api_key = api_key or os.environ.get("COS_API_KEY_ID")
        self.service_crn = service_crn or os.environ.get("COS_INSTANCE_CRN")
        self.endpoint = endpoint or os.environ.get("COS_ENDPOINT")
        self.verify_ssl = verify_ssl

        # Initializes the IBM COS (Cloud Object Storage) client
        self.cos_client = ibm_boto3.client(
            "s3",
            ibm_api_key_id=self.api_key,
            ibm_service_instance_id=self.service_crn,
            endpoint_url=self.endpoint,
            verify=self.verify_ssl,
            config=Config(signature_version="oauth")
        )

        self._initialized = True

    def _format_key(self, folder: str, year: str, filename: str) -> str:
        # Builds a structured key/path for storage based on folder/year/filename
        folder = folder.strip("/ ")
        year = year.strip("/ ")
        filename = filename.strip("/ ")
        return f"{folder}/{year}/{filename}" if folder else filename


    def object_exists(self, bucket: str, key: str) -> bool:
        """Checks if a given object (file) exists in the specified bucket."""
        try:
            self.cos_client.head_object(Bucket=bucket, Key=key)
            return True
        except ClientError as e:
            if e.response["Error"]["Code"] == "404":
                return False
            raise


    def upload_file(self, file_path: str, bucket: str, filename: str, year: str = "", folder: str = "", overwrite: bool = True) -> bool:
        """Uploads a file from the local filesystem to IBM COS (Cloud Object Storage)."""
        key = self._format_key(folder, year, filename)
        if not overwrite and self.object_exists(bucket, key):
            print(f"[!] File {key} already exists in {bucket} (overwrite=False)")
            return False
        try:
            self.cos_client.upload_file(file_path, bucket, key)
            print(f"File uploaded: {bucket}/{key}")
            return True
        except ClientError as e:
            print(f"Error uploading file from disk: {e}")
            return False


    def upload_file_stream(self, stream: BytesIO, bucket: str, filename: str, year: str = "", folder: str = "", overwrite: bool = True) -> bool:
        """Uploads a file from an in-memory stream (BytesIO) to IBM COS."""
        key = self._format_key(folder, year, filename)
        if not overwrite and self.object_exists(bucket, key):
            print(f"File {key} already exists in {bucket} (overwrite=False)")
            return False
        try:
            self.cos_client.upload_fileobj(stream, Bucket=bucket, Key=key)
            print(f"Stream file uploaded: {bucket}/{key}")
            return True
        except ClientError as e:
            print(f"Error uploading file from memory: {e}")
            return False


    def get_object_stream(self, bucket: str, key: str) -> BytesIO:
        """Retrieves an object from IBM COS as a BytesIO stream."""
        try:
            response = self.cos_client.get_object(Bucket=bucket, Key=key)
            return BytesIO(response["Body"].read())
        except ClientError as e:
            print(f"Error retrieving file: {e}")
            return None
        
        
    def download_file(self, bucket: str, key: str, destination_path: str) -> bool:
        """Download a file from COS to the local file system"""
        try:
            os.makedirs(os.path.dirname(destination_path), exist_ok=True)
            self.cos_client.download_file(Bucket=bucket, Key=key, Filename=destination_path)
            print(f"Downloaded {bucket}/{key} -> {destination_path}")
            return True
        except ClientError as e:
            print(f"Error downloading file: {e}")
            return False


    def delete_object(self, bucket: str, key: str) -> bool:
        """Deletes an object from the specified bucket."""
        try:
            self.cos_client.delete_object(Bucket=bucket, Key=key)
            print(f"Object deleted: {bucket}/{key}")
            return True
        except ClientError as e:
            print(f"Error deleting file: {e}")
            return False


    def list_objects(self, bucket: str, prefix: str = ""):
        """Lists objects stored in a specific folder (prefix) in the bucket."""
        try:
            response = self.cos_client.list_objects(Bucket=bucket, Prefix=prefix)
            return [obj["Key"] for obj in response.get("Contents", [])]
        except ClientError as e:
            print(f"Error listing objects: {e}")
            return []


In [30]:
# Constants for IBM COS values
COS_ENDPOINT = "https://s3.us-south.cloud-object-storage.appdomain.cloud"
COS_API_KEY_ID = "1IVfOmwvWWvLchoC2reuQNIoCC_s28ri3k3jQ30PI1XW"
COS_INSTANCE_CRN = "crn:v1:bluemix:public:cloud-object-storage:global:a/a3ddc697dfc74d418f1d2974f79fe20e:a106584d-8374-4867-ad78-7b73f6bf2966::"

In [31]:
# from src.services.ibm_cos_manager import IBMCOSManager
from datetime import datetime
# import os

cos = IBMCOSManager(
    api_key=COS_API_KEY_ID, 
    service_crn=COS_INSTANCE_CRN,
    endpoint=COS_ENDPOINT
)


In [32]:
# Después de guardar localmente como file_path_data
#filename = f"WSP-{datetime.now().year}-{list_report[k]}.pdf"
folder = "temp"  # o carpetaB según lógica
bucket = "bucket-synapse-iq"

In [33]:
cos.list_objects(bucket,"temp")

['temp/2025/prueba002.pdf',
 'temp/2025/prueba003.pdf',
 'temp/2025/prueba004.pdf',
 'temp/2025/prueba005.pdf',
 'temp/prueba002.pdf']

In [34]:
# Subida desde disco
file_path_data = "/Users/cristianb/Documents/Python/rel8ed/SynapseIQ_staging/storage/kansas/004321.pdf"
cos.upload_file(
    file_path=file_path_data,
    bucket=bucket,
    filename="004321.pdf",
    year="2025",
    folder="kansas",
    overwrite=True
)


File uploaded: bucket-synapse-iq/kansas/2025/004321.pdf


True

In [37]:
# O, alternativa: Subida desde memoria
with open("/Users/cristianb/Documents/Python/rel8ed/SynapseIQ_staging/storage/kansas/004370.pdf", "rb") as f:
    stream = BytesIO(f.read())

cos.upload_file_stream(
    stream=stream,
    bucket=bucket,
    filename="prueba005.pdf",
    year="2025",
    folder="temp",
    overwrite=True
)

Stream file uploaded: bucket-synapse-iq/temp/2025/prueba005.pdf


True

In [39]:
cos.list_objects(bucket)

['Texas Good Ones.pdf',
 'kansas/004321.pdf',
 'kansas/2025/004321.pdf',
 'output.md',
 'temp/2025/2025/prueba005.pdf',
 'temp/2025/prueba002.pdf',
 'temp/2025/prueba003.pdf',
 'temp/2025/prueba004.pdf',
 'temp/2025/prueba005.pdf',
 'temp/prueba002.pdf']

# Text Estraction

In [ ]:
from ibm_watsonx_ai.metanames import TextExtractionsMetaNames
from ibm_watsonx_ai.helpers import DataConnection, S3Location
from ibm_watsonx_ai import Credentials
from ibm_watsonx_ai.foundation_models import ModelInference
from ibm_watsonx_ai import APIClient
from ibm_watsonx_ai.foundation_models.extractions import TextExtractions

credentials = Credentials(
    url = "https://us-south.ml.cloud.ibm.com",
    api_key = "ZGSeCv9jm4AW0jhCbIWQUl7WeBU4ZYIYL0iCwSVoDISS",
)

client = APIClient(credentials)

model_id = "meta-llama/llama-3-2-11b-vision-instruct"
project_id = "fa52e12e-affe-4ea5-a08c-60f6fe58174d"


extraction = TextExtractions(api_client=client,
                             project_id=project_id)

def extract():
    document_reference = DataConnection(connection_asset_id="7b3b547d-0655-4518-959c-d41cbf643271",
                                        location=S3Location(bucket="bucket-synapse-iq",
                                                            path="Texas Good Ones.pdf"))
    results_reference = DataConnection(connection_asset_id="7b3b547d-0655-4518-959c-d41cbf643271",
                                    location=S3Location(bucket="bucket-synapse-iq",
                                                        path="output.md"))

    TextExtractionsMetaNames().show()
    print(TextExtractionsMetaNames().get_example_values())

    steps = {TextExtractionsMetaNames.OCR: {'languages_list': ['en']},
            TextExtractionsMetaNames.TABLE_PROCESSING: {'enabled': True}}


    #task_credentials_details = client.task_credentials.store()

    #client.task_credentials.list()

    details = extraction.run_job(document_reference=document_reference,
                                results_reference=results_reference,
                                steps=steps,
                                results_format="markdown")
    extraction_job_id = extraction.get_id(extraction_details=details)
    print(extraction_job_id)
    print(details)
    return details

# Watson CLient

In [ ]:
from ibm_watsonx_ai import APIClient
from ibm_watsonx_ai import Credentials
from ibm_watsonx_ai.foundation_models import ModelInference
import sys

credentials = Credentials(
    url = "https://us-south.ml.cloud.ibm.com",
    api_key = "ZGSeCv9jm4AW0jhCbIWQUl7WeBU4ZYIYL0iCwSVoDISS"
)

client = APIClient(credentials)

params = {
    "time_limit": 10000,
    "max_new_token": 100
}

model_id = "meta-llama/llama-3-2-11b-vision-instruct"
project_id = "2696de87-406d-40ce-8fcc-07693ac2740b"
space_id = None # optional
verify = False

model = ModelInference(
  model_id=model_id,
  api_client=client,
  params=params,
  project_id=project_id,
  space_id=space_id,
  verify=verify,
)

def invokeModel():
  messages = [
    {
      "role": "system",
      "content": """You are a helpful assistant who extracts the "NAME OF PASSENGER" from a pdf. 
                   """
    },
    {
      "role": "user",
      "content": [
        {
          "type": "text",
          "text":'https://www.flhsmv.gov/pdf/forms/90011s.pdf'
        }
      ]
     },
    # {
    #   "role": "assistant",
    #   "content": "The distance between Paris, France, and Bangalore, India, is approximately 7,800 kilometers (4,850 miles)"
    # }
  ]

  data = {
      "role": "user",
      "content": [
        {
          "type": "text",
          "text": "what are the fields ion this pdf"
        }
      ]
    }
  
  messages.append(data)
  return model.chat(messages=messages) 

print()
print(invokeModel())


# Subir Registros Pasados

In [1]:
import os
from io import BytesIO
import ibm_boto3
from ibm_botocore.client import Config, ClientError
from threading import Lock
import pandas as pd

class IBMCOSManager:
    _instance = None
    _lock = Lock()

    def __new__(cls, *args, **kwargs):
        # Singleton pattern: ensures only one instance is created
        if not cls._instance:
            with cls._lock:
                if not cls._instance:
                    cls._instance = super(IBMCOSManager, cls).__new__(cls)
        return cls._instance

    def __init__(self,
                 api_key=None,
                 service_crn=None,
                 endpoint=None,
                 verify_ssl=True):

        # Prevents re-initialization of the singleton instance
        if hasattr(self, "_initialized") and self._initialized:
            return

        self.api_key = api_key or os.environ.get("COS_API_KEY_ID")
        self.service_crn = service_crn or os.environ.get("COS_INSTANCE_CRN")
        self.endpoint = endpoint or os.environ.get("COS_ENDPOINT")
        self.verify_ssl = verify_ssl

        # Initializes the IBM COS (Cloud Object Storage) client
        self.cos_client = ibm_boto3.client(
            "s3",
            ibm_api_key_id=self.api_key,
            ibm_service_instance_id=self.service_crn,
            endpoint_url=self.endpoint,
            verify=self.verify_ssl,
            config=Config(signature_version="oauth")
        )

        self._initialized = True

    def _format_key(self, folder: str, year: str, filename: str) -> str:
        # Builds a structured key/path for storage based on folder/year/filename
        folder = folder.strip("/ ")
        year = year.strip("/ ")
        filename = filename.strip("/ ")
        return f"{folder}/{year}/{filename}" if folder else filename


    def object_exists(self, bucket: str, key: str) -> bool:
        """Checks if a given object (file) exists in the specified bucket."""
        try:
            self.cos_client.head_object(Bucket=bucket, Key=key)
            return True
        except ClientError as e:
            if e.response["Error"]["Code"] == "404":
                return False
            raise


    def upload_file(self, file_path: str, bucket: str, filename: str, year: str = "", folder: str = "", overwrite: bool = True) -> bool:
        """Uploads a file from the local filesystem to IBM COS (Cloud Object Storage)."""
        key = self._format_key(folder, year, filename)
        if not overwrite and self.object_exists(bucket, key):
            print(f"[!] File {key} already exists in {bucket} (overwrite=False)")
            return False
        try:
            self.cos_client.upload_file(file_path, bucket, key)
            print(f"File uploaded: {bucket}/{key}")
            return True
        except ClientError as e:
            print(f"Error uploading file from disk: {e}")
            return False


    def upload_file_stream(self, stream: BytesIO, bucket: str, filename: str, year: str = "", folder: str = "", overwrite: bool = True) -> bool:
        """Uploads a file from an in-memory stream (BytesIO) to IBM COS."""
        key = self._format_key(folder, year, filename)
        if not overwrite and self.object_exists(bucket, key):
            print(f"File {key} already exists in {bucket} (overwrite=False)")
            return ""
        try:
            self.cos_client.upload_fileobj(stream, Bucket=bucket, Key=key)
            print(f"Stream file uploaded: {bucket}/{key}")
            return f"{bucket}/{key}"
        except ClientError as e:
            print(f"Error uploading file from memory: {e}")
            return ""


    def get_object_stream(self, bucket: str, key: str) -> BytesIO:
        """Retrieves an object from IBM COS as a BytesIO stream."""
        try:
            response = self.cos_client.get_object(Bucket=bucket, Key=key)
            return BytesIO(response["Body"].read())
        except ClientError as e:
            print(f"Error retrieving file: {e}")
            return None
        
        
    def download_file(self, bucket: str, key: str, destination_path: str) -> bool:
        """Download a file from COS to the local file system"""
        try:
            os.makedirs(os.path.dirname(destination_path), exist_ok=True)
            self.cos_client.download_file(Bucket=bucket, Key=key, Filename=destination_path)
            print(f"Downloaded {bucket}/{key} -> {destination_path}")
            return True
        except ClientError as e:
            print(f"Error downloading file: {e}")
            return False

    def upload_dataframe_to_cos_csv(self, df: pd.DataFrame, bucket: str, 
                                    filename: str, year: str = "", folder: str = "", 
                                    overwrite: bool = True) -> bool:
        """
        Converts a DataFrame into CSV format and uploads it to IBM COS in memory.

        Parameters:
        - df: DataFrame to upload
        - bucket: Destination bucket name in IBM COS
        - filename: Name of the CSV file (e.g., 'report.csv')
        - folder: Logical folder path inside the bucket (e.g., 'reports/2025')
        - overwrite: Whether to overwrite the file if it already exists

        Returns:
        - The location where the file was uploaded if the upload was successful, "" otherwise
        """
        try:
            # Convert to CSV in memory
            buffer = BytesIO()
            df.to_csv(buffer, index=False)
            buffer.seek(0)

            # Subir con la clase IBM COS
            return self.upload_file_stream(
                stream=buffer,
                bucket=bucket,
                filename=filename,
                year=year,
                folder=folder,
                overwrite=overwrite
            )
        except Exception as e:
            print(f"Error loading DataFrame as CSV: {e}")
            return ""




    def delete_object(self, bucket: str, key: str) -> bool:
        """Deletes an object from the specified bucket."""
        try:
            self.cos_client.delete_object(Bucket=bucket, Key=key)
            print(f"Object deleted: {bucket}/{key}")
            return True
        except ClientError as e:
            print(f"Error deleting file: {e}")
            return False


    def list_objects(self, bucket: str, prefix: str = ""):
        """Lists objects stored in a specific folder (prefix) in the bucket."""
        try:
            response = self.cos_client.list_objects(Bucket=bucket, Prefix=prefix)
            return [obj["Key"] for obj in response.get("Contents", [])]
        except ClientError as e:
            print(f"Error listing objects: {e}")
            return []


In [ ]:


# === Configuración general
local_folder_base = "/Users/cristianb/Documents/Python/rel8ed/SynapseIQ_staging/storage/"  # carpeta donde están los subdirectorios por estado
bucket_name = "bucket-synapse-iq"
year = "2025"

# === Lista de estados que quieres subir
states = ["ohio", "kansas", "winstonsalem", "minnesota"]  # puedes recorrer todos los subdirectorios si prefieres

# === Inicializar cliente COS
cos = IBMCOSManager()

for state in states:
    local_dir = os.path.join(local_folder_base, state)
    remote_folder = state  # será usado como carpeta lógica en COS

    if not os.path.isdir(local_dir):
        print(f" Carpeta no encontrada: {local_dir}")
        continue

    for file_name in os.listdir(local_dir):
        if not file_name.lower().endswith(".pdf"):
            continue

        local_path = os.path.join(local_dir, file_name)

        print(f" Subiendo {local_path} -> COS: /{remote_folder}/{year}/{file_name}")
        cos.upload_file(
            file_path=local_path,
            bucket=bucket_name,
            filename=file_name,
            year=year,
            folder=remote_folder,
            overwrite=True  # cambia a False si no quieres reemplazar existentes
        )
